In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import re
from collections import Counter

In [2]:
# df_prof = pd.read_csv("dataset-exemplos.csv", sep=";")

df_prof = pd.read_csv("dataset-10k.csv", sep=";")


In [3]:
df_prof.head(500)

,ID,Text,Label
0,D-7077,The renaissance is a great example of how seem...,OpenAI
1,D-8361,Quantum mechanics sits at an interesting inter...,Anthropic
2,D-5510,The renaissance describes a set of phenomena i...,Google
3,D-1838,"There's a lot of debate around obesity, and I ...",Human
4,D-4923,The internet of things is a well-established a...,Google
...,...,...,...
495,D-7401,One of the most compelling things about coloni...,OpenAI
496,D-746,"When you think about vaccines, it's easy to ge...",Human
497,D-2749,Research on cybersecurity identifies zero-day ...,Meta
498,D-6164,"When exploring the Roman Empire, it helps to b...",OpenAI


In [4]:
stopwords = {
    # Artigos e determinantes
    "a", "an", "the", "this", "that", "these", "those", "some", "any",
    "each", "every", "either", "neither", "no", "both", "all", "few",
    "more", "most", "other", "such", "what", "which", "who", "whoever",
    "whatever", "whichever",

    # Preposições
    "in", "on", "at", "by", "for", "with", "about", "against", "between",
    "into", "through", "during", "before", "after", "above", "below",
    "from", "up", "down", "out", "off", "over", "under", "again",
    "further", "then", "once", "upon", "within", "without", "along",
    "following", "across", "behind", "beyond", "plus", "except",
    "until", "toward", "among", "onto", "per", "versus", "via",
    "near", "since", "around", "throughout",

    # Conjunções
    "and", "or", "but", "if", "while", "although", "because", "since",
    "unless", "until", "whereas", "whether", "though", "even", "yet",
    "so", "nor", "neither", "both", "either", "not", "also", "however",

    # Pronomes
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "who", "whom", "whose", "anyone", "someone", "everyone", "nobody",
    "one", "ones",

    # Verbos auxiliares e cópula
    "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having",
    "do", "does", "did", "doing",
    "will", "would", "shall", "should",
    "may", "might", "must", "can", "could",
    "need", "dare", "ought", "used",

    # Advérbios comuns
    "not", "no", "very", "too", "just", "also", "only", "now", "then",
    "here", "there", "when", "where", "why", "how", "often", "always",
    "never", "sometimes", "usually", "already", "still", "yet", "ever",
    "soon", "rather", "quite", "nearly", "almost", "enough", "much",
    "many", "well", "back", "even", "away", "perhaps", "maybe",
    "indeed", "therefore", "thus", "hence", "otherwise", "instead",
    "meanwhile", "moreover", "furthermore", "nevertheless", "nonetheless",
    "accordingly", "consequently", "subsequently",

    # Outros funcionais
    "of", "to", "as", "than", "so", "due", "via", "re", "eg", "ie",
    "etc", "vs", "get", "got", "let", "like", "make", "made", "say",
    "said", "new", "old", "way", "part", "take", "come", "go", "see",
    "know", "think", "look", "use", "find", "give", "tell", "ask",
    "seem", "feel", "try", "call", "keep", "put", "set", "work",
    "provide", "include", "involve", "contain", "refer", "show",
    "appear", "become", "remain", "result", "require", "allow",
    "follow", "represent", "consider",
}

In [5]:
def simple_stem(word):

    suffixes = ["ing","ed","ly","s"]

    for suffix in suffixes:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[:-len(suffix)]

    return word

In [7]:
def preprocess_text(text):

    text = str(text)
    text = text.lower()

    # remover html
    text = re.sub(r"<.*?>", "", text)

    # remover números
    text = re.sub(r"\d+", "", text)

    # remover pontuação
    text = re.sub(r"[^\w\s]", "", text)

    # remover espaços extra
    text = re.sub(r"\s+", " ", text).strip()

    # remover caracteres não alfabéticos
    text = re.sub(r"[^a-zA-Z ]", " ", text)

    # remover stopwords e stemmatizar
    # words = text.split()
    # words = [w for w in words if w not in stopwords]
    words = text.split()
    words = [simple_stem(w) for w in words if w not in stopwords]

    return " ".join(words)

In [8]:
# Limpar texto do dataset

df_prof["Text"] = df_prof["Text"].apply(preprocess_text)

In [9]:
print(df_prof.columns)

Index(['ID', 'Text', 'Label'], dtype='object')


In [10]:
# Divisão do dataset em treino e teste (80/20)

df = df_prof.sample(frac=1, random_state=42).reset_index(drop=True)

test_size = 0.2
split_index = int(len(df) * (1 - test_size))

train_df = df[:split_index]
test_df = df[split_index:]

print("Treino:", len(train_df))
print("Teste:", len(test_df))

Treino: 8000
Teste: 2000


Até aqui temos:
✔ dataset carregado
✔ preprocessamento definido
✔ preprocessamento aplicado
✔ split treino/teste
---------------------------------------------------------------------
---------------------------------------------------------------------

In [11]:
# Separar features (X) e labels (y)

X_train = train_df["Text"].values
y_train = train_df["Label"].values

X_test = test_df["Text"].values
y_test = test_df["Label"].values

print(X_train[:5])
print(y_train[:5])
print(X_test[:5])
print(y_test[:5])

['gut microbiome encompasse range relat concept notab dysbiosi microbial diversity principle intestinal bacteria serve unify framework interpret experimental observational data metabolic function introduce additional complexity researcher continue investigate together element define scope significance gut microbiome contemporary science technology'
 'ive found neuroplasticity subject reward curiosity people dont realise central neural pathway whole picture understand brain adaptation play surprising important role doesnt attention there someth poetic cortical remapp cognitive rehabilitation connect that make neuroplasticity worth explor'
 'understand organ transplantation require engag genuine uncertainty relationship graft survival rejection episode wellestablish respect important question immunosuppression treat settl worth acknowledg evidence stronger context other wait list add complexity current model dont ful capture intellectual honesty limit seem important discuss organ transpl

Converter labels para números

As redes neuronais não trabalham com texto, por isso precisamos transformar:

   Human
   GPT
   Claude

em algo como:

   0
   1
   2

In [12]:
labels = sorted(set(y_train))

label_to_id = {label: i for i, label in enumerate(labels)}

y_train = np.array([label_to_id[label] for label in y_train])
y_test = np.array([label_to_id[label] for label in y_test])

print(label_to_id)

{'Anthropic': 0, 'Google': 1, 'Human': 2, 'Meta': 3, 'OpenAI': 4}


In [13]:
# Criar vocabulário (usando só treino)

vocab = {}

for text in X_train:

    words = text.split()

    for word in words:

        if word not in vocab:
            vocab[word] = len(vocab)

print("Tamanho do vocabulário:", len(vocab))

Tamanho do vocabulário: 866


In [14]:
# Contar frequência de palavras no conjunto de treino

from collections import Counter

word_counts = Counter()

for text in X_train:
    words = text.split()
    word_counts.update(words)

In [15]:
max_vocab = 1500

most_common = word_counts.most_common(max_vocab)

vocab = {word:i for i,(word,count) in enumerate(most_common)}

print("Vocabulário reduzido:", len(vocab))

Vocabulário reduzido: 866


In [17]:
# Função Bag-of-Words
# Função que transforma texto em números

def text_to_bow(text, vocab):

    vector = np.zeros(len(vocab))

    words = text.split()

    for word in words:

        if word in vocab:
            index = vocab[word]
            vector[index] += 1

    return vector

# teste da função - pode ser removido depois
print(text_to_bow("spermidine naturally living Carlos", vocab))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [18]:
# Criar matriz de treino e teste

X_train_bow = np.array([text_to_bow(text, vocab) for text in X_train])
X_test_bow = np.array([text_to_bow(text, vocab) for text in X_test])

print("X_train_bow shape:", X_train_bow.shape)
print("X_test_bow shape:", X_test_bow.shape)

X_train_bow shape: (8000, 866)
X_test_bow shape: (2000, 866)


In [19]:
print(X_train_bow[:2])

[[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]]


In [20]:
X_train_bow = X_train_bow / np.maximum(X_train_bow.sum(axis=1, keepdims=True), 1)
X_test_bow = X_test_bow / np.maximum(X_test_bow.sum(axis=1, keepdims=True), 1)

Até aqui temos:
✔ separação das features (Text) e labels (Label)
✔ conversão das labels de texto para valores numéricos (label encoding manual)
✔ construção do vocabulário usando apenas os dados de treino
✔ implementação manual do modelo Bag-of-Words
✔ transformação dos textos de treino e teste em vetores numéricos
✔ preprocessamento definido
✔ preprocessamento aplicado
✔ split treino/teste
---------------------------------------------------------------------
---------------------------------------------------------------------

In [21]:
# One-Hot Encoding das labels

def one_hot(y, num_classes):

    y_encoded = np.zeros((len(y), num_classes))

    for i in range(len(y)):
        y_encoded[i, y[i]] = 1

    return y_encoded

In [22]:
# Aplicar One-Hot Encoding das labels

num_classes = len(label_to_id)

y_train_oh = one_hot(y_train, num_classes)
y_test_oh = one_hot(y_test, num_classes)

print(y_train_oh.shape)

# NOTA: resultado (100, 5) significa 100 amostras, 5 classes.

(8000, 5)


In [23]:
# Criar a primeira Dense Layer
# Agora começa a parte mais importante da tarefa: implementar a rede neural sem bibliotecas de ML.

# class DenseLayer:

#     # def __init__(self, input_size, output_size):

#     #     self.weights = np.random.randn(input_size, output_size) * 0.01
#     #     self.bias = np.zeros((1, output_size))

#     def __init__(self, input_size, output_size):

#         self.weights = np.random.randn(input_size, output_size) * np.sqrt(2.0 / input_size)  # He init
#         self.bias = np.zeros((1, output_size))

#     def forward(self, input):

#         self.input = input
#         return np.dot(input, self.weights) + self.bias

#     def backward(self, grad_output, learning_rate):

#         grad_weights = np.dot(self.input.T, grad_output)
#         grad_bias = np.sum(grad_output, axis=0, keepdims=True)
#         grad_input = np.dot(grad_output, self.weights.T)

#         self.weights -= learning_rate * grad_weights
#         self.bias -= learning_rate * grad_bias

#         return grad_input
    
class DenseLayer:
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * np.sqrt(2.0 / input_size)  # He init
        self.bias = np.zeros((1, output_size))

    def forward(self, input):
        self.input = input
        return np.dot(input, self.weights) + self.bias

    def backward(self, grad_output, learning_rate, l2=0.001):
        grad_weights = np.dot(self.input.T, grad_output) + l2 * self.weights
        grad_bias = np.sum(grad_output, axis=0, keepdims=True)
        grad_input = np.dot(grad_output, self.weights.T)
        self.weights -= learning_rate * grad_weights
        self.bias -= learning_rate * grad_bias
        return grad_input
    # Esta classe representa uma camada totalmente ligada (fully connected layer).

In [24]:
class Dropout:
    def __init__(self, rate=0.3):
        self.rate = rate
        self.mask = None

    def forward(self, input, training=True):
        if not training:
            return input
        self.mask = (np.random.rand(*input.shape) > self.rate) / (1 - self.rate)
        return input * self.mask

    def backward(self, grad_output, learning_rate):
        return grad_output * self.mask

In [25]:
# Função de ativação ReLU

def relu(x):
    return np.maximum(0, x)

def relu_prime(x):
    return (x > 0).astype(float)

In [26]:
# Camada de activação

class Activation:

    def __init__(self, activation, activation_prime):

        self.activation = activation
        self.activation_prime = activation_prime

    def forward(self, input):

        self.input = input
        return self.activation(input)

    def backward(self, grad_output, learning_rate):

        return grad_output * self.activation_prime(self.input)

In [27]:
# Softmax Camada de saída para classificação multi-classe

class SoftmaxLayer:

    def forward(self, input):

        exp_values = np.exp(input - np.max(input, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

        return self.output

    def backward(self, grad_output, learning_rate):

        return grad_output
    

# Softmax transforma os outputs em probabilidades.
# Exemplo:  [2.3, 0.5, 1.1] → [0.68, 0.09, 0.23]

In [28]:
# Classe de rede neural

# class NeuralNetwork:

#     def __init__(self):
#         self.layers = []

#     def add(self, layer):
#         self.layers.append(layer)

#     def forward(self, X):

#         output = X

#         for layer in self.layers:
#             output = layer.forward(output)

#         return output

#     def backward(self, grad, learning_rate):

#         for layer in reversed(self.layers):
#             grad = layer.backward(grad, learning_rate)

class NeuralNetwork:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X, training=False):
        output = X
        for layer in self.layers:
            if isinstance(layer, Dropout):
                output = layer.forward(output, training=training)
            else:
                output = layer.forward(output)
        return output

    def backward(self, grad, learning_rate):
        for layer in reversed(self.layers):
            grad = layer.backward(grad, learning_rate)

In [29]:
# Função de treino

# def train(network, X, y, epochs, learning_rate):

    # for epoch in range(epochs):

    #     output = network.forward(X)

    #     loss_grad = output - y

    #     network.backward(loss_grad, learning_rate)

    #     if epoch % 10 == 0:

    #         loss = -np.mean(np.sum(y * np.log(output + 1e-8), axis=1))

    #         print("Epoch", epoch, "Loss:", loss)

# def train(network, X, y, epochs, learning_rate):

#     n = len(X)

#     for epoch in range(epochs):

#         for i in range(n):

#             x = X[i:i+1]
#             target = y[i:i+1]

#             output = network.forward(x)

#             grad = output - target

#             network.backward(grad, learning_rate)

#         if epoch % 10 == 0:

#             output = network.forward(X)

#             loss = -np.mean(np.sum(y * np.log(output + 1e-8), axis=1))

#             print("Epoch", epoch, "Loss:", loss)

# def train(network, X, y, epochs, learning_rate):
#     n = len(X)
#     for epoch in range(epochs):
#         for i in range(n):
#             x = X[i:i+1]
#             target = y[i:i+1]
#             output = network.forward(x, training=True)  # ← training=True
#             grad = output - target
#             network.backward(grad, learning_rate)

#         if epoch % 10 == 0:
#             output = network.forward(X)  # training=False para avaliar
#             loss = -np.mean(np.sum(y * np.log(output + 1e-8), axis=1))
#             print("Epoch", epoch, "Loss:", loss)

def train(network, X, y, epochs, learning_rate, batch_size=16):
    n = len(X)
    for epoch in range(epochs):
        # shuffle
        indices = np.random.permutation(n)
        X_shuffled = X[indices]
        y_shuffled = y[indices]

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            x_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]

            output = network.forward(x_batch, training=True)
            grad = (output - y_batch) / len(x_batch)
            network.backward(grad, learning_rate)

        if epoch % 10 == 0:
            output = network.forward(X)
            loss = -np.mean(np.sum(y * np.log(output + 1e-8), axis=1))
            print("Epoch", epoch, "Loss:", loss)

In [30]:
# Construir a rede neural

input_size = X_train_bow.shape[1]
output_size = num_classes

# nn = NeuralNetwork()

# nn.add(DenseLayer(input_size, 64))
# nn.add(Activation(relu, relu_prime))

# nn.add(DenseLayer(64, output_size))
# nn.add(SoftmaxLayer())

nn = NeuralNetwork()
nn.add(DenseLayer(input_size, 64))
nn.add(Activation(relu, relu_prime))
nn.add(Dropout(0.3))
nn.add(DenseLayer(64, output_size))
nn.add(SoftmaxLayer())

# train(nn, X_train_bow, y_train_oh, epochs=400, learning_rate=0.01)

# Arquitetura:
#
#     input
#      ↓
#     Dense(64)
#      ↓
#     ReLU
#      ↓
#     Dense(classes)
#      ↓
#     Softmax

In [31]:
# Treinar a rede neural

# train(nn, X_train_bow, y_train_oh, epochs=200, learning_rate=0.1)
train(nn, X_train_bow, y_train_oh, epochs=200, learning_rate=0.05, batch_size=16)

Epoch 0 Loss: 1.5780282645674697
Epoch 10 Loss: 0.2956884466656175
Epoch 20 Loss: 0.08191760629125718
Epoch 30 Loss: 0.06322920212823227
Epoch 40 Loss: 0.059684427301755105
Epoch 50 Loss: 0.05972545363189445
Epoch 60 Loss: 0.05923785954760154
Epoch 70 Loss: 0.05948245351760852
Epoch 80 Loss: 0.05945056875009692
Epoch 90 Loss: 0.0599403630905444
Epoch 100 Loss: 0.05931252472146265
Epoch 110 Loss: 0.060005155152947236
Epoch 120 Loss: 0.05987098175247584
Epoch 130 Loss: 0.059895622040762236
Epoch 140 Loss: 0.05960505860018689
Epoch 150 Loss: 0.059821993498150774
Epoch 160 Loss: 0.06037039405549501
Epoch 170 Loss: 0.06104892305168352
Epoch 180 Loss: 0.06015327439818951
Epoch 190 Loss: 0.06019608587872999


In [281]:
def predict(network, X):

    output = network.forward(X)

    return np.argmax(output, axis=1)

In [282]:
# Avaliar o modelo

y_pred = predict(nn, X_test_bow)

accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [283]:
print(nn.forward(X_test_bow[:5]))

[[0.01340574 0.94447633 0.01239481 0.01535554 0.01436758]
 [0.01988834 0.012162   0.94422395 0.01046275 0.01326296]
 [0.94545017 0.01643783 0.01754478 0.00874218 0.01182504]
 [0.94555472 0.01843716 0.01636404 0.00754427 0.01209982]
 [0.01769004 0.01854749 0.01699613 0.01394448 0.93282186]]
